# Zadanie 3: optymalizacja dyskretna

Termin realizacji: 20 kwietnia 2026

Zadanie do oddania przez MS Teams. Do oddania: kod oraz krótkie sprawozdanie w PDF (można na przykład przy użyciu `quarto render notebook.ipynb --to pdf`).

## Na 3.0

Do realizacji:

1. Zaimplementuj dyskretny problem plecakowy z trzema plecakami w MiniZinc na podstawie przykładu (plik `minizinc.ipynb`). Spróbuj rozwiązać problem dla 10 zestawów parametrów o różnych wielkościach tak, aby rozwiązanie największego problemu trwało powyżej 5 sekund. Zanotuj w każdym przypadku liczbę wszystkich przedmiotów, pojemności plecaków, liczbę wybranych przedmiotów i sumaryczną wartość przedmiotów w każdym plecaku osobno.
2. Losowanie przedmiotów w każdym przypadku powinno być tak ustawione, aby optymalne rozwiązanie wymagało wzięcia przynajmniej dwóch przedmiotów do każdego plecaka oraz zostawienia przynajmniej dwóch przedmiotów poza plecakami. W raporcie należy umieścić informację w jaki sposób zostało to zweryfikowane.
3. Zmodyfikuj metodę z notatnika `tabu_search.ipynb` tak aby rozwiązywała opisywany problem plecakowy. Porównaj na tych samych problemach czy Minizinc i Tabu search zwracają równie dobre rozwiazania, oraz wypisz jakie to są rozwiązania. Wykonaj eksperymenty z trzema różnymi długościami listy zakazów (1, 2, 5).

## Na 4.0

Do realizacji:

1. Punkty z zadania na 3.0.
2. Rozszerz możliwe ruchy w tabu search o przeniesienie przedmiotu z jednego plecaka do drugiego. Zapisz rozważ czy to poprawia działanie metody (czy znalezione jest lepsze, takie samo czy gorsze rozwiązanie? czy rozwiązanie jest znajdowane szybciej czy wolniej?). Dla każdego z 10 zestawów parametrów problemu plecakowego wykonaj ocenę przez uśrednienie dla 10 różnych losowych przypadków.
3. Podsumuj dane w formie tabelki z czterema kolumnami (Minizinc, tabu search z listą o długości 1, 2, i 5) oraz 10 wierszami (po jednym dla zestawu parametrów problemu), a w komórkach umieść średnią wartość wartości przedmiotów oraz średni czas potrzebny do uzyskania rozwiązania.

## Na 5.0

Do realizacji:

1. Punkty z zadania na 4.0.
2. Zaimplementuj samodzielnie algorytm symulowanego wyżarzania z podobnym interfejsem co tabu search. Porównaj jego działanie do rozważanych wcześniej rozwiązań dla trzech różnych schematów chłodzenia. Dobierz liczbę iteracji tak, aby czas działania był porównywalny do tabu search.


# Na 3.0

## Generacja 10 losowych przypadków
### Parametry:
* 3 plecaki
* 10 zestawów parametrów
* największy - conajmniej 10 sek.
* optymalne rozwiązanie: 
    * kady plecak ma pojemośc 2 razy lub więcej większą od wagi kazdego przedmiotu
    * suma pojemnosci plecakow < suma wag - 2 najwieksze wagi

In [ ]:
# import Pkg; Pkg.add("Distributions")
# import Pkg; Pkg.add("DataStructures")

   Resolving package versions...
   Installed DataStructures ─ v0.19.4
    Updating `~/.julia/environments/v1.12/Project.toml`
  [864edb3b] + DataStructures v0.19.4 [loaded: v0.19.3]
    Updating `~/.julia/environments/v1.12/Manifest.toml`
  [864edb3b] ↑ DataStructures v0.19.3 ⇒ v0.19.4 [loaded: v0.19.3]
Precompiling packages...
   1239.1 ms  ✓ DataStructures
  10730.4 ms  ✓ Distributions → DistributionsTestExt
  2 dependencies successfully precompiled in 13 seconds. 252 already precompiled.
  1 dependency precompiled but a different version is currently loaded. Restart julia to access the new version. Otherwise, loading dependents of this package may trigger further precompilation to work with the unexpected version.


In [1]:
using Distributions

#     gościu sugeruje, że losować 3 kategorie przedmiotów - małe waggi i małe wartości, średnie wagi, i duże wagi i że naturalnie te constrainty się spełnią
#     do modelu w minizincu dodać funckje sprawdzające czy roziwązanie ma spełnione te zasady i dodać je jako zmienne do rozwiązania
#     eksperymenty
#     minizinc api do julii - istnieje!!

function make_dzn(n::Int, n_knapsacks::Int)
    profits = rand(DiscreteUniform(10, 1000), n)
    weights = rand(DiscreteUniform(10, 100), n)
    capacities = rand(DiscreteUniform(100, 300), n_knapsacks)
    content = """
    ITEM = _(1..$n);
    knapsack_n = $n_knapsacks;
    capacities = $capacities;
    profits = $profits;
    weights = $weights;
    """
    file = open("knapsack_generated.dzn", "w+")
    write(file, content)
    close(file)
end
make_dzn(100, 3)


SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up


In [2]:
struct KnapsackProblem
    capacities::Vector{Int}
    weights::Vector{Int}
    profits::Vector{Int}
end
# Zrobić zamiast tego wczytywanie z pliku .dzn
function generate_problem(n_knapsacks = 3, n_items = 100)
    capacities = rand(DiscreteUniform(2500, 3000), n_knapsacks)
    profits = rand(DiscreteUniform(10, 1000), n_items)
    weights = rand(DiscreteUniform(10, 100), n_items)
    kp = KnapsackProblem(capacities, profits, weights)
end

kp1 = generate_problem()

KnapsackProblem([2573, 2906, 2814], [179, 698, 894, 198, 646, 814, 622, 160, 480, 119  …  597, 449, 370, 386, 631, 995, 858, 90, 719, 302], [75, 13, 26, 95, 83, 69, 91, 33, 83, 78  …  57, 46, 59, 34, 41, 74, 98, 67, 21, 67])

## TabuSearch

In [5]:
using DataStructures
using Distributions

mutable struct TabuState{TMove,P,TF}
    tabu_buffer::CircularBuffer{TMove}
    best_seen::P
    best_seen_obj::TF
    current::P
    considered::P
    iter::Int
end

function TabuState(p, x0; buffer_length::Int=10)
    moves = possible_moves(p, x0)
    obj = objective(p, x0)
    return TabuState{eltype(moves),typeof(x0),typeof(obj)}(
        CircularBuffer{eltype(moves)}(buffer_length), x0, obj, copy(x0), copy(x0), 1
    )
end


function solve_tabu(p, s::TabuState; iteration_limit::Int=100)
    iter_best_move = 0
    while s.iter < iteration_limit
        moves = possible_moves(p, s.current)
        best_move = 0
        best_move_obj = Inf
        for (i_move, move) in enumerate(moves)
            if in(move, s.tabu_buffer)
                # move forbidden, do not consider
                continue
            end
            # evaluate move
            copyto!(s.considered, s.current)
            apply!(s.considered, move)
            considered_value = objective(p, s.considered)
            if considered_value < best_move_obj
                best_move = i_move
                best_move_obj = considered_value
            end
        end
        # no allowed move found
        if best_move == 0
            break
        end
        chosen_move = moves[best_move]
        item_id = chosen_move[1]
        prev_bin = s.current[item_id]
        apply!(s.current, chosen_move)
        push!(s.tabu_buffer, invert_move(p, moves[best_move], prev_bin))
        if best_move_obj < s.best_seen_obj
            # best so far, let's remember it
            iter_best_move = s.iter
            copyto!(s.best_seen, s.current)
            s.best_seen_obj = best_move_obj
        end
        s.iter += 1
    end
    println("Iteracja w której zostało osiągnięte najlepsze rozwiązanie: ", iter_best_move)
    return s.best_seen
end


function objective(p::KnapsackProblem, x::Vector{Int})
    profit = 0
    for i in eachindex(x)
        if x[i] > 0
            profit += p.profits[i]
        end
    end
    return -profit
end


function apply!(x, move::Tuple{Int,Int})
    item_idx, new_bin = move
    x[item_idx] = new_bin
    return x
end

function invert_move(::KnapsackProblem, move::Tuple{Int,Int}, old_bin::Int)
    return (move[1], old_bin)
end


function possible_moves(p::KnapsackProblem, x::Vector{Int})
    move_list = Tuple{Int,Int}[]
    current_weights = zeros(Int, length(p.capacities)) #sum(p.weights .* x)
    # add item
    for (item_id, bin_id) in enumerate(x)
        if bin_id > 0 # 0 oznacza brak przypisania
            current_weights[bin_id] += p.weights[item_id]
        end
    end    

    for i in eachindex(x)
        if x[i] > 0
            # możliwość usunięcia i z plecaka x[i]
            push!(move_list, (i, 0))
        end
        for k in eachindex(p.capacities)
            # przeniesienie do plecaka
            if x[i] == 0 && (current_weights[k] + p.weights[i] <= p.capacities[k])
                push!(move_list, (i,k))
            end
        end
    end

    return move_list
end


possible_moves (generic function with 1 method)

In [6]:
function test(kp)
    x0 = fill(0, length(kp.weights))
    st = TabuState(kp, x0; buffer_length=10)
    sol = solve_tabu(kp, st; iteration_limit=1000000)

    println("Pełne rozwiązanie: ", sol)
    
    println("Best objective: ", st.best_seen_obj)
    println("Last iteration: ", st.iter)
end
@time begin
test(kp1)
end



Iteracja w której zostało osiągnięte najlepsze rozwiązanie: 29
Pełne rozwiązanie: [3, 0, 0, 2, 0, 0, 0, 0, 0, 2, 0, 3, 0, 0, 0, 0, 0, 0, 0, 2, 0, 2, 0, 0, 0, 3, 0, 0, 0, 3, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 2, 0, 1, 0, 0, 0, 2, 0, 0, 0, 2, 0, 1, 0, 3, 0, 0, 0, 0, 0, 0, 3, 2, 0, 2, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 2, 0, 0]
Best objective: -2095
Last iteration: 1000000
  2.946175 seconds (6.99 M allocations: 951.365 MiB, 2.40% gc time, 10.00% compilation time)


### 3.3 - rózne długości linii zakazów

In [7]:
function experiment(kp)
    dlugosci = [1, 2, 5]
    
    for L in dlugosci
        @time begin
        println("\n--- Eksperyment dla długości tabu L = $L ---")
        
        x0 = fill(0, length(kp.weights))
        # Tutaj przekazujemy L do parametru buffer_length
        st = TabuState(kp, x0; buffer_length = L)
        
        sol = solve_tabu(kp, st; iteration_limit = 100000)
        
        println("Pełne rozwiązanie: ", sol)
    
        println("Best objective: ", st.best_seen_obj)
        println("Last iteration: ", st.iter)
        end
    end
end
experiment(kp1)


--- Eksperyment dla długości tabu L = 1 ---
Iteracja w której zostało osiągnięte najlepsze rozwiązanie: 25
Pełne rozwiązanie: [3, 0, 0, 2, 0, 0, 3, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 2, 0, 0, 0, 0, 0, 0, 0, 3, 1, 1, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 2, 0, 0, 0, 0, 0, 1, 0, 0, 0, 2, 0, 1, 0, 0, 0, 1, 0, 3, 0, 0, 0, 0, 0, 0, 3, 2, 0, 2, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0]
Best objective: -1969
Last iteration: 100000
  0.254118 seconds (609.38 k allocations: 90.682 MiB, 4.24% gc time, 3.39% compilation time)

--- Eksperyment dla długości tabu L = 2 ---
Iteracja w której zostało osiągnięte najlepsze rozwiązanie: 22
Pełne rozwiązanie: [3, 0, 0, 2, 0, 0, 3, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 2, 0, 0, 0, 0, 0, 0, 0, 3, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 2, 0, 0, 0, 0, 0, 1, 0, 0, 0, 2, 0, 0, 0, 0, 0, 1, 0, 3, 0, 0, 0, 0, 0, 0, 3, 2, 0, 2, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0]
Best o

## MiniZinc

# Na 4.0

In [8]:
function possible_moves(p::KnapsackProblem, x::Vector{Int})
    move_list = Tuple{Int,Int}[]
    current_weights = zeros(Int, length(p.capacities)) #sum(p.weights .* x)
    # add item
    for (item_id, bin_id) in enumerate(x)
        if bin_id > 0 # 0 oznacza brak przypisania
            current_weights[bin_id] += p.weights[item_id]
        end
    end    

    for i in eachindex(x)
        if x[i] > 0
            # możliwość usunięcia i z plecaka x[i]
            push!(move_list, (i, 0))
        end
        for k in eachindex(p.capacities)
            # przeniesienie do plecaka
            if x[i] == 0 && (current_weights[k] + p.weights[i] <= p.capacities[k])
                push!(move_list, (i,k))
            end
            # dodatkowa możliwość - przeniesienie do innego plecaka, do punktu 4.0
            if x[i] != k && (current_weights[k] + p.weights[i] <= p.capacities[k])
                push!(move_list, (i, k))
            end
        end
    end

    return move_list
end

possible_moves (generic function with 1 method)

## Na 5.0

## Symulowane wyzazanie - TabuSearch

In [9]:
mutable struct SAState{P, TF}
    current::P
    best_seen::P
    best_seen_obj::TF
    temp::Float64
    iter::Int
end
# Konstruktor pomocniczy
function SAState(p, x0; initial_temp::Float64=100.0)
    obj = objective(p, x0)
    return SAState{typeof(x0), typeof(obj)}(
        copy(x0),
        copy(x0),
        obj,
        initial_temp,
        1
    )
end
function solve_simulated_annealing(p, s::SAState; 
                                   max_iter=10000, 
                                   cooling_rate=0.995)
    iter_best_move = 0
    while s.iter < max_iter

        moves = possible_moves(p, s.current)
        if isempty(moves) break end
        move = rand(moves) 
        old_obj = objective(p, s.current)
        
        test_state = copy(s.current)
        apply!(test_state, move)
        new_obj = objective(p, test_state)
        
        ΔE = new_obj - old_obj
        
        if ΔE < 0 || rand() < exp(-ΔE / s.temp)
            s.current = test_state
            
            if new_obj < s.best_seen_obj
                iter_best_move = s.iter
                s.best_seen = copy(s.current)
                s.best_seen_obj = new_obj
            end
        end
        
        s.temp *= cooling_rate
        s.iter += 1
    end
    println("Iteracja w której zostało osiągnięte najlepsze rozwiązanie: ", iter_best_move)
    return s.best_seen
end

solve_simulated_annealing (generic function with 1 method)

In [10]:

function test2(kp)
    x0 = fill(0, length(kp.weights))
    st = SAState(kp, x0)
    sol = solve_simulated_annealing(kp, st; max_iter=1000000)

    println("Full solution: ", sol)
    
    println("Best objective: ", st.best_seen_obj)
    println("Last iteration: ", st.iter)
end
@time begin
test2(kp1)
end

Iteracja w której zostało osiągnięte najlepsze rozwiązanie: 328
Full solution: [2, 0, 0, 2, 0, 0, 0, 1, 0, 2, 0, 1, 2, 2, 0, 1, 1, 0, 0, 2, 0, 2, 0, 0, 0, 1, 0, 3, 0, 0, 0, 3, 2, 0, 0, 0, 0, 0, 1, 0, 3, 0, 3, 2, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 3, 2, 2, 3, 2, 0, 0, 1, 2, 3, 0, 2, 1, 0, 3, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 3]
Best objective: -2218
Last iteration: 1000000
  1.659607 seconds (10.13 M allocations: 4.164 GiB, 13.38% gc time, 3.66% compilation time)
